# SQL Analysis using SQLite


## 1. Building the database and loading the cleaned data

In [1]:
import sqlite3
from pathlib import Path

import pandas as pd

BASE = Path.cwd()
if BASE.name == "notebooks":
    BASE = BASE.parent
CLEANED = BASE / "data" / "cleaned"
SQL_DIR = BASE / "sql"
DB_PATH = BASE / "db" / "ecommerce.db"
RESULTS = BASE / "output" / "query_results"
RESULTS.mkdir(parents=True, exist_ok=True)
DB_PATH.parent.mkdir(parents=True, exist_ok=True)

conn = sqlite3.connect(DB_PATH)
conn.executescript((SQL_DIR / "schema.sql").read_text(encoding="utf-8"))

for table in ["customers", "products", "orders", "order_items"]:
    df = pd.read_csv(CLEANED / f"{table}.csv")
    df.to_sql(table, conn, if_exists="append", index=False)
    print(f"{table:12s} {len(df):>5} rows loaded")

conn.commit()

customers      800 rows loaded
products       500 rows loaded
orders        1997 rows loaded
order_items   4977 rows loaded


In [2]:
def run_query(name, preview=None):
    sql = (SQL_DIR / f"{name}.sql").read_text(encoding="utf-8")
    df = pd.read_sql_query(sql, conn)
    df.to_csv(RESULTS / f"{name}.csv", index=False)
    return df.head(preview) if preview else df

## Q1 (Basic) - Total revenue per category


In [3]:
run_query("01_revenue_per_category")

,category,total_revenue
0,Home,2129440.19
1,Electronics,1928323.64
2,Books,1861235.12
3,Clothing,1648087.53


## Q2 (Basic) - Top 10 customers by total order value

In [4]:
run_query("02_top10_customers")

,customer_id,customer_name,customer_type,num_orders,total_order_value
0,C0175,Haley Arnold,PREMIUM,7,39957.83
1,C0432,Andrew Gonzales,REGULAR,5,39481.54
2,C0366,Chloe Tran,REGULAR,4,37804.97
3,C0230,Charles Martinez,VIP,6,34934.99
4,C0025,Pamela Romero,VIP,5,34920.24
5,C0160,Katherine Davis,REGULAR,6,34843.71
6,C0591,Jason Cortez,PREMIUM,6,34696.93
7,C0041,Nicholas Mcbride,PREMIUM,6,33984.31
8,C0110,Daniel Murphy,VIP,5,33791.28
9,C0516,Omar Ware,PREMIUM,6,33504.64


## Q3 (Basic) - Month-wise order count, last 12 months


In [5]:
run_query("03_monthly_order_count")

,order_month,order_count
0,2025-07,46
1,2025-08,78
2,2025-09,76
3,2025-10,109
4,2025-11,87
5,2025-12,112
6,2026-01,113
7,2026-02,109
8,2026-03,154
9,2026-04,161


## Q4 (Intermediate) - Customers who ordered but never had anything delivered


In [6]:
run_query("04_customers_never_delivered")

,customer_id,customer_name,total_orders,statuses_seen
0,C0504,David Ortega,6,"SHIPPED,PLACED,RETURNED,CANCELLED"
1,C0276,Lisa Cox,5,"PLACED,RETURNED,SHIPPED"
2,C0432,Andrew Gonzales,5,"CANCELLED,RETURNED,SHIPPED"
3,C0679,Kara Jackson,5,"SHIPPED,CANCELLED,PLACED"
4,C0111,Sandra King,4,"SHIPPED,PLACED"
...,...,...,...,...
168,C0766,Sarah Rivera,1,CANCELLED
169,C0775,Amber Mcconnell,1,RETURNED
170,C0786,Ryan Stokes,1,PLACED
171,C0790,Jesse Jensen,1,SHIPPED


## Q5 (Intermediate) - Products with more returns than purchases


In [7]:
run_query("05_more_returns_than_purchases")

,product_id,product_name,category,units_purchased,units_returned
0,P0439,Hansen Premium Scarf,Clothing,26,62
1,P0481,Lee Ultra Hoodie,Clothing,29,65
2,P0462,Haynes Classic Webcam,Electronics,38,66


## Q6 (Intermediate) - Return rate per category


In [8]:
run_query("06_return_rate_per_category")

,category,units_purchased,units_returned,return_rate_percent
0,Clothing,3407,182,5.07
1,Electronics,3526,152,4.13
2,Books,3693,104,2.74
3,Home,3790,81,2.09


## Q7 (Advanced) - Running revenue total per region

In [9]:
run_query("07_running_total_by_region", preview=15)

,region_code,order_date,daily_revenue,running_total
0,CENTRAL,2024-07-06,4730.68,4730.68
1,CENTRAL,2024-07-18,6233.16,10963.84
2,CENTRAL,2024-07-31,4788.61,15752.45
3,CENTRAL,2024-08-01,5911.82,21664.28
4,CENTRAL,2024-08-19,4584.03,26248.30
5,CENTRAL,2024-08-23,3407.96,29656.26
6,CENTRAL,2024-08-28,13489.15,43145.42
7,CENTRAL,2024-09-04,3665.73,46811.15
8,CENTRAL,2024-09-09,442.06,47253.21
9,CENTRAL,2024-09-18,1945.41,49198.62


## Q8 (Advanced) - Product revenue rank within category (DENSE_RANK)


In [10]:
run_query("08_dense_rank_products", preview=15)

,category,product_name,total_revenue,rank_in_category
0,Books,Brown Vintage Guidebook,49037.64,1
1,Books,Bell Modern Guidebook,40447.57,2
2,Books,Turner Pro Textbook,38100.77,3
3,Books,Fernandez Eco Guidebook,35799.42,4
4,Books,Hardin Pro Cookbook,35162.05,5
5,Books,Baker Ultra Encyclopedia,32810.41,6
6,Books,Moore Compact Textbook,32394.36,7
7,Books,James Premium Guidebook,32006.14,8
8,Books,Walters Compact Atlas,31985.63,9
9,Books,Moore Deluxe Biography,30879.93,10


## Q9 (Advanced) - Days between consecutive orders (LAG) + At-Risk flag


In [11]:
run_query("09_lag_days_between_orders", preview=15)

,customer_id,order_date,previous_order_date,days_gap,avg_days_gap,risk_flag
0,C0001,2024-09-19 17:44:37,NaN,NaN,359.7,At Risk
1,C0001,2025-09-14 11:19:18,2024-09-19 17:44:37,359.7,359.7,At Risk
2,C0003,2024-10-04 00:21:26,NaN,NaN,211.2,At Risk
3,C0003,2025-03-23 03:21:28,2024-10-04 00:21:26,170.1,211.2,At Risk
4,C0003,2026-05-01 07:30:45,2025-03-23 03:21:28,404.2,211.2,At Risk
5,C0003,2026-06-29 11:32:37,2026-05-01 07:30:45,59.2,211.2,At Risk
6,C0004,2025-10-15 04:52:41,NaN,NaN,NaN,Active
7,C0005,2025-10-22 23:37:26,NaN,NaN,67.0,At Risk
8,C0005,2025-11-01 15:37:31,2025-10-22 23:37:26,9.7,67.0,At Risk
9,C0005,2025-11-25 07:49:03,2025-11-01 15:37:31,23.7,67.0,At Risk


## Q10 (Advanced) - Multi-level CTE: monthly revenue → High/Medium/Low buckets


In [12]:
run_query("10_cte_customer_categories", preview=15)

,order_month,revenue_category,customer_count
0,2024-07,High,1
1,2024-07,Medium,4
2,2024-07,Low,7
3,2024-08,High,1
4,2024-08,Medium,3
5,2024-08,Low,13
6,2024-09,Medium,4
7,2024-09,Low,11
8,2024-10,High,1
9,2024-10,Medium,8


## Q11 (Advanced) - NTILE(4) lifetime-value quartiles


In [13]:
run_query("11_ntile_quartiles", preview=12)

,customer_id,total_value,quartile,quartile_label
0,C0175,39957.83,1,Platinum
1,C0432,39481.54,1,Platinum
2,C0366,37804.97,1,Platinum
3,C0230,34934.99,1,Platinum
4,C0025,34920.24,1,Platinum
5,C0160,34843.71,1,Platinum
6,C0591,34696.93,1,Platinum
7,C0041,33984.31,1,Platinum
8,C0110,33791.28,1,Platinum
9,C0516,33504.64,1,Platinum


## Q12 (Advanced) - Year-over-Year monthly revenue


In [14]:
run_query("12_yoy_comparison")

,year,month,revenue,prev_year_revenue,yoy_growth_percent
0,2024,7,54517.24,NaN,NaN
1,2024,8,56876.85,NaN,NaN
2,2024,9,61904.94,NaN,NaN
3,2024,10,92065.99,NaN,NaN
4,2024,11,88777.64,NaN,NaN
5,2024,12,132187.67,NaN,NaN
6,2025,1,119207.83,NaN,NaN
7,2025,2,153369.67,NaN,NaN
8,2025,3,152083.97,NaN,NaN
9,2025,4,197252.77,NaN,NaN


## Q13 (Advanced) - First vs most recent purchased category


In [15]:
run_query("13_first_last_category", preview=15)

,customer_id,first_category,most_recent_category,category_shift
0,C0001,Clothing,Books,Yes
1,C0003,Books,Electronics,Yes
2,C0004,Electronics,Clothing,Yes
3,C0005,Electronics,Electronics,No
4,C0006,Clothing,Clothing,No
5,C0008,Electronics,Books,Yes
6,C0009,Books,Electronics,Yes
7,C0010,Books,Clothing,Yes
8,C0011,Books,Home,Yes
9,C0012,Books,Home,Yes


## Q14 (Advanced) - Cumulative revenue distribution (Pareto)


In [16]:
run_query("14_cumulative_distribution", preview=15)

,customer_id,revenue,cumulative_revenue,cumulative_percent,customer_percentile
0,C0175,39957.83,39957.83,0.55,0.14
1,C0432,39481.54,79439.38,1.10,0.28
2,C0366,37804.97,117244.34,1.63,0.42
3,C0230,34934.99,152179.34,2.11,0.56
4,C0025,34920.24,187099.57,2.59,0.71
5,C0160,34843.71,221943.29,3.08,0.85
6,C0591,34696.93,256640.21,3.56,0.99
7,C0041,33984.31,290624.52,4.03,1.13
8,C0110,33791.28,324415.80,4.50,1.27
9,C0516,33504.64,357920.44,4.96,1.41


## Q15 (Advanced) - Cohort retention by registration month


In [17]:
run_query("15_cohort_analysis")

,cohort_month,cohort_customers,month_0,month_1,month_2,month_3,retention_m0_pct,retention_m1_pct,retention_m2_pct,retention_m3_pct
0,2024-01,17,0,0,0,0,0.0,0.0,0.0,0.0
1,2024-02,26,0,0,0,0,0.0,0.0,0.0,0.0
2,2024-03,27,0,0,0,0,0.0,0.0,0.0,0.0
3,2024-04,31,0,0,0,3,0.0,0.0,0.0,9.7
4,2024-05,28,0,0,3,2,0.0,0.0,10.7,7.1
5,2024-06,28,0,2,2,2,0.0,7.1,7.1,7.1
6,2024-07,22,0,0,3,2,0.0,0.0,13.6,9.1
7,2024-08,28,1,2,0,4,3.6,7.1,0.0,14.3
8,2024-09,18,1,1,1,1,5.6,5.6,5.6,5.6
9,2024-10,17,1,1,2,4,5.9,5.9,11.8,23.5


## Q16 (Advanced) - Products frequently bought together


In [18]:
run_query("16_bought_together", preview=15)

,product_a,product_b,times_bought_together
0,Bell Modern Guidebook,Gordon Ultra Coffee Maker,3
1,Gray Deluxe Hoodie,Smith Eco Blazer,3
2,Ryan Eco Mouse,Davis Premium Wall Clock,3
3,Anderson Classic Laptop,Winters Premium Smartphone,2
4,Armstrong Deluxe Bookshelf,Galvan Modern Blazer,2
5,Armstrong Deluxe Bookshelf,Lane Compact Cap,2
6,Atkinson Ultra Jacket,Carr Ultra Textbook,2
7,Ayala Ultra T-Shirt,Brennan Smart Encyclopedia,2
8,Ballard Classic Router,Bowers Ultra Blazer,2
9,Banks Compact Wall Clock,Mclaughlin Pro Guidebook,2


## Saving all 16 queries


In [19]:
saved = sorted(RESULTS.glob("*.csv"))
print(f"{len(saved)} result files:")
for f in saved:
    print(" ", f.name)
conn.close()

16 result files:
  01_revenue_per_category.csv
  02_top10_customers.csv
  03_monthly_order_count.csv
  04_customers_never_delivered.csv
  05_more_returns_than_purchases.csv
  06_return_rate_per_category.csv
  07_running_total_by_region.csv
  08_dense_rank_products.csv
  09_lag_days_between_orders.csv
  10_cte_customer_categories.csv
  11_ntile_quartiles.csv
  12_yoy_comparison.csv
  13_first_last_category.csv
  14_cumulative_distribution.csv
  15_cohort_analysis.csv
  16_bought_together.csv
